In [ ]:
import numpy as np
import pandas as pd
from scipy.io import savemat
import os
import matlab.engine

def generate_time_sequence(length):
    return [i / 96 for i in range(length)]

def create_experiment_kla_sequence(days, nominal_kla, kla_value, DR_len, tank=3):
    steps_per_day = 96
    n_ininominal = 32
    n_DR = DR_len * 4  # Convert hours to 15-minute steps
    n_fnlnominal = steps_per_day - n_ininominal - n_DR
    day_pattern = [nominal_kla] * n_ininominal + [kla_value] * n_DR + [nominal_kla] * n_fnlnominal
    return day_pattern * days

def save_kla_to_mat(kla_sequence, tank, tag):
    time_seq = generate_time_sequence(len(kla_sequence))
    df = pd.DataFrame({'Sequence': time_seq, 'Value': kla_sequence})
    combined = df[['Sequence', 'Value']].values.tolist()

    var_name = f'KLa{tank}_Setpoints_ASM3'
    filename = f'ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3/KLa{tank}_Setpoints_{tag}.mat'
    savemat(filename, {var_name: combined})
    print(f"Saved: {filename}")

def create_experimental_kla_files(experiment_length, days=609):
    """Create experimental KLa files for all tanks with given experiment length"""
    kla_nominal_value = 240
    kla_experiment_value = 0
    
    print(f"\nCreating experimental KLa files for experiment_length = {experiment_length} hours")
    
    for tank in [3, 4, 5]:
        # Generate experimental sequence
        experiment_seq = create_experiment_kla_sequence(
            days, kla_nominal_value, kla_experiment_value, experiment_length, tank=tank
        )
        
        # Save experimental .mat file
        save_kla_to_mat(experiment_seq, tank=tank, tag='experiment')
    
    print(f"✓ Completed experimental KLa files for {experiment_length}-hour DR events")

def run_ASM3_data_generation():
    """Run the ASM3 DR data generation MATLAB script"""
    print("\nStarting MATLAB ASM3 data generation...")
    
    try:
        # Start MATLAB engine
        eng = matlab.engine.start_matlab()
        
        # Change to the correct directory (update this path as needed)
        eng.cd('ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3', nargout=0)
        
        # Run the simplified ASM3 DR data generation script
        eng.run('ASM3_DR_datagen.m', nargout=0)
        
        # Quit MATLAB
        eng.quit()
        
        print("✓ MATLAB data generation completed successfully")
        
    except Exception as e:
        print(f"✗ Error during MATLAB execution: {str(e)}")
        raise

def backup_results(experiment_length):
    """Backup results for current experiment length"""
    backup_dir = f"Results_ExpLength_{experiment_length}h"
    
    try:
        if not os.path.exists(backup_dir):
            os.makedirs(backup_dir)
        
        import shutil
        
        # Copy ASM3_OutputDB directory contents to backup
        source_output_dir = "ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3/ASM3_OutputDB"
        if os.path.exists(source_output_dir):
            shutil.copytree(source_output_dir, f"{backup_dir}/ASM3_OutputDB", dirs_exist_ok=True)
            print(f"✓ ASM3_OutputDB backed up to {backup_dir}/ASM3_OutputDB")
        else:
            print(f"⚠ Warning: Output directory {source_output_dir} not found")
        
        # Copy DR_images directory contents to backup
        source_images_dir = "ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3/DR_images"
        if os.path.exists(source_images_dir):
            shutil.copytree(source_images_dir, f"{backup_dir}/DR_images", dirs_exist_ok=True)
            print(f"✓ DR_images backed up to {backup_dir}/DR_images")
        else:
            print(f"⚠ Warning: Images directory {source_images_dir} not found")
            
        print(f"✓ Complete results backup completed for {experiment_length}h experiment")
            
    except Exception as e:
        print(f"⚠ Warning: Could not backup results: {str(e)}")


In [ ]:

def main():
    """Main automation routine for multiple DR dataset generation"""
    
    print("=" * 60)
    print("AUTOMATED DR DATASETS GENERATION")
    print("=" * 60)
    print("Generating datasets for experiment lengths: 4, 5, 6, 7 hours")
    print("Total datasets to generate: 4")
    print("=" * 60)
    
    # Configuration
    start_length = 4
    end_length = 7
    total_datasets = end_length - start_length + 1
    
    for i, experiment_length in enumerate(range(start_length, end_length + 1), 1):
        
        print(f"\n{'='*50}")
        print(f"DATASET {i}/{total_datasets}: {experiment_length}-HOUR DR EVENTS")
        print(f"{'='*50}")
        
        try:
            # Step 1: Create experimental KLa input files
            create_experimental_kla_files(experiment_length)
            
            # Step 2: Run MATLAB data generation
            run_ASM3_data_generation()
            
            # Step 3: Backup results
            backup_results(experiment_length)
            
            print(f"\n✓ Dataset {i}/{total_datasets} completed successfully")
            
        except Exception as e:
            print(f"\n✗ Error in dataset {i}: {str(e)}")
            print("Stopping automation due to error.")
            break
    
    print(f"\n{'='*60}")
    print("AUTOMATION COMPLETED")
    print(f"{'='*60}")
    print(f"Generated {i} datasets with experiment lengths from {start_length} to {experiment_length} hours")
    print("Results are backed up in separate directories for each experiment length")
    print("Check individual Results_ExpLength_*h directories for outputs")

if __name__ == "__main__":
    main()

AUTOMATED DR DATASETS GENERATION
Generating datasets for experiment lengths: 4, 5, 6, 7 hours
Total datasets to generate: 4

DATASET 1/4: 4-HOUR DR EVENTS

Creating experimental KLa files for experiment_length = 4 hours
Saved: ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3/KLa3_Setpoints_experiment.mat
Saved: ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3/KLa4_Setpoints_experiment.mat
Saved: ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3/KLa5_Setpoints_experiment.mat
✓ Completed experimental KLa files for 4-hour DR events

Starting MATLAB ASM3 data generation...

=== ASM3 BENCHMARK MODEL ===
Initializing workspace and model parameters...
Configuration complete:
  - Simulation duration: 609.0 days
  - Calibration period: 245.0 days
  - Segment duration: 14.0 days
  - Total segments: 26

=== PHASE 1b: STEADY STATE INITIALIZATION ===
Running steady state model with constant influent...
✓ Steady state initialization completed
  States saved to workspace_steady_state_initial.mat

=== PHASE 1a: EXPERIMENTAL CALI